In [ ]:
import os

# 1. 强制清理旧残留
print("正在清理旧文件...")
!rm -rf Diffusion-Illusions
!rm -rf master.zip

# 2. 克隆仓库
print("正在克隆仓库...")
!git clone https://github.com/RyannDaGreat/Diffusion-Illusions

# 3. 检查是否成功
if os.path.exists('Diffusion-Illusions'):
    print("✅ 仓库克隆成功！")
    
    # 4. 进入目录
    %cd Diffusion-Illusions
    
    # 5. 安装依赖
    print("正在安装依赖 (红色警告请忽略)...")
    !pip install -r requirements.txt
    !pip install mediapy easydict "numpy<2.0"
    
    print("\n✅✅ 环境初始化全部完成！")
    print("⚠️⚠️ 现在的关键步骤：请点击上方菜单 'Runtime' -> 'Restart session' 重启运行时！")
    
else:
    print("❌❌ 克隆还是失败了，请检查网络。")

In [ ]:
import os
repo_name = "Diffusion-Illusions"
if os.getcwd().endswith(repo_name):
    print(f"✅ 当前位置正确: {os.getcwd()}")
else:
    if os.path.exists(repo_name):
        %cd {repo_name}
        print(f"✅ 已切换工作目录到: {os.getcwd()}")
    else:
        print("⚠️ 文件夹不存在，正在重新克隆...")
        !git clone https://github.com/RyannDaGreat/Diffusion-Illusions
        %cd {repo_name}
        print(f"✅ 克隆并切换完成: {os.getcwd()}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import rp
from typing import List, Tuple
import source.stable_diffusion as sd
from source.stable_diffusion_labels import SimpleLabel
from source.learnable_textures import LearnableImageFourier, LearnableImageRasterSigmoided

# --- 辅助函数 ---
def gaussian_kernel_2d(size: int, sigma: float) -> np.ndarray:
    return rp.gaussian_kernel(size=size, sigma=sigma, dim=2)

def apply_filter_single_channel(channel: torch.Tensor, kernel: torch.Tensor, device: str = 'cuda') -> torch.Tensor:
    kernel = kernel.to(device).unsqueeze(0).unsqueeze(0)
    padding = kernel.shape[-1] // 2
    channel_batch = channel.unsqueeze(0).unsqueeze(0)
    filtered = F.conv2d(channel_batch, kernel, padding=padding)
    return filtered.squeeze(0).squeeze(0)

# --- 核心模型类 ---
class LearnableColorChannelHybridSD(nn.Module):
    def __init__(self, cutoffs_low, cutoffs_high, size=512, representation='fourier', device='cuda'):
        super().__init__()
        self.device = device
        # 共享模式：两张图，分别贡献低频和高频
        if representation == 'fourier':
            self.learnable_far = LearnableImageFourier(size, size, 3)
            self.learnable_close = LearnableImageFourier(size, size, 3)
        else:
            self.learnable_far = LearnableImageRasterSigmoided(size, size, 3)
            self.learnable_close = LearnableImageRasterSigmoided(size, size, 3)
        
        self.filters_low = []
        self.filters_high = []
        
        for cutoff_low, cutoff_high in zip(cutoffs_low, cutoffs_high):
            sigma_low = size / (2 * np.pi * cutoff_low)
            sigma_high = size / (2 * np.pi * cutoff_high)
            kernel_size = min(int(6 * max(sigma_low, sigma_high)) | 1, 99)
            
            low_pass = torch.tensor(gaussian_kernel_2d(kernel_size, sigma_low), dtype=torch.float32).to(self.device)
            high_pass = torch.tensor(gaussian_kernel_2d(kernel_size, sigma_high), dtype=torch.float32).to(self.device)
            self.filters_low.append(low_pass)
            self.filters_high.append(high_pass)
    
    def forward(self) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        img_far = self.learnable_far()
        img_close = self.learnable_close()
        
        channels = []
        for i in range(3):
            # 远看提取低频，近看提取高频
            low_freq = apply_filter_single_channel(img_far[i], self.filters_low[i], self.device)
            high_freq_base = apply_filter_single_channel(img_close[i], self.filters_high[i], self.device)
            high_freq = img_close[i] - high_freq_base
            
            channel = low_freq + high_freq
            channel = torch.clamp(channel, 0, 1)
            channels.append(channel)
        
        hybrid = torch.stack(channels, dim=0)
        return hybrid, img_far, img_close

In [ ]:
if 'model_sd' not in dir():
    print("正在加载 Stable Diffusion...")
    model_name = "CompVis/stable-diffusion-v1-4"
    gpu = rp.select_torch_device()
    model_sd = sd.StableDiffusion(gpu, model_name)
    device = model_sd.device
    print("✅ 模型加载完毕！")
else:
    print("模型已存在。")

In [ ]:
from IPython.display import clear_output

# ===========================
# 🔧 参数配置
# ===========================
# 1. 远看的内容 (RGB 三通道)
prompts_far = [
    "vibrant red passionate energy",  # R 通道
    "cheerful yellow happiness",      # G 通道
    "romantic pink love"              # B 通道
]
# 2. 近看的内容 (RGB 三通道)
prompts_close = [
    "calm gray peaceful serenity",    # R 通道
    "cool blue tranquil waters",      # G 通道
    "deep purple mysterious mood"     # B 通道
]

# 通道频率控制
cutoffs_low = [6, 8, 10]
cutoffs_high = [24, 28, 32]

NUM_ITER = 3000
LEARNING_RATE = 1e-4
GUIDANCE_SCALE = 100
DISPLAY_INTERVAL = 100

# ===========================
# 🚀 训练循环
# ===========================
print("初始化 RGB 通道分离任务...")

model = LearnableColorChannelHybridSD(
    cutoffs_low=cutoffs_low, 
    cutoffs_high=cutoffs_high,
    size=512, 
    representation='fourier',
    device=device
).to(device)

# 组合 Prompt 进行整体训练
label_far_combined = SimpleLabel(", ".join(prompts_far))
label_close_combined = SimpleLabel(", ".join(prompts_close))
optim = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

display_eta = rp.eta(NUM_ITER, title='Training Status')

try:
    for iter_num in range(NUM_ITER):
        display_eta(iter_num)
        
        hybrid, img_far, img_close = model()
        
        # Loss 1: 远看图像训练 (低频部分)
        model_sd.train_step(label_far_combined.embedding, img_far[None], guidance_scale=GUIDANCE_SCALE)
        
        # Loss 2: 近看图像训练 (高频部分)
        model_sd.train_step(label_close_combined.embedding, img_close[None], guidance_scale=GUIDANCE_SCALE)
        
        optim.step()
        optim.zero_grad()
        
        if iter_num % DISPLAY_INTERVAL == 0:
            clear_output(wait=True)
            with torch.no_grad():
                hybrid, far, close = model()
                v_hybrid = rp.as_numpy_image(hybrid)
                v_far = rp.as_numpy_image(far)
                v_close = rp.as_numpy_image(close)
                
                print(f"Iteration {iter_num} / {NUM_ITER}")
                print("左: 最终混合图 | 中: 远看效果 | 右: 近看效果")
                rp.display_image(rp.tiled_images([v_hybrid, v_far, v_close]))

except KeyboardInterrupt:
    print("⚠️ 用户停止训练")

print("✅ 最终结果 (右键保存):")
with torch.no_grad():
    final_hybrid, _, _ = model()
    rp.display_image(rp.as_numpy_image(final_hybrid))